# Traffic Collision Data Analysis

# Objective

In this case study, you will be working on California Traffic Collision Data Analysis using Apache Spark, a powerful distributed computing framework designed for big data processing. This assignment aims to provide hands-on experience in analyzing large-scale traffic collision datasets using PySpark and AWS services. You will apply data analytics techniques to clean, transform, and explore crash data, drawing meaningful insights to support traffic safety and urban planning. Beyond understanding how big data tools optimize performance on a single machine and across clusters, you will develop a structured approach to analyzing crash trends, identifying high-risk locations, and evaluating contributing factors to traffic incidents. Additionally, you will utilize AWS S3 to store the processed data efficiently after the ETL process, enabling scalable storage and easy retrieval for further analysis.


# Business Value:

Traffic collisions pose significant risks to public safety, requiring continuous monitoring and analysis to enhance road safety measures. Government agencies, city planners, and policymakers must leverage data-driven insights to improve infrastructure, optimize traffic management, and implement preventive measures.

In this assignment, you will analyze California traffic collision data to uncover patterns related to accident severity, location-based risks, and key contributing factors. With Apache Spark's ability to handle large datasets efficiently and AWS S3's scalable storage, transportation authorities can process vast amounts of crash data in real time, enabling faster and more informed decision-making.

As an analyst examining traffic safety trends, your task is to analyze historical crash data to derive actionable insights that can drive policy improvements and safety interventions. Your analysis will help identify high-risk areas, categorize accidents by severity and contributing factors, and store the processed data in an AWS S3 bucket for scalable and long-term storage.

By leveraging big data analytics and cloud-based storage, urban planners and traffic authorities can enhance road safety strategies, reduce accident rates, and improve public transportation planning.


# Dataset Overview

The dataset used in this analysis consists of California traffic collision data obtained from the Statewide Integrated Traffic Records System (SWITRS). It includes detailed records of traffic incidents across California, covering various attributes such as location, severity, involved parties, and contributing factors. The dataset has been preprocessed and transformed using PySpark to facilitate large-scale analysis. By leveraging Apache Spark, we ensure efficient data handling, enabling deeper insights into traffic patterns, accident trends, and potential safety improvements.

The dataset is a .sqlite file contains detailed information about traffic collisions across California and is structured into four primary tables:
- `collisions` table contains information about the collision, where it happened, what vehicles were involved.

- `parties` table contains information about the groups people involved in the collision including age, sex, and sobriety.

- `victims` table contains information about the injuries of specific people involved in the collision.

- `locations` table contains information about the geographical location and details of road intersections.

# Analysis Workflow
<ol>
    <li>
        <strong>Data Preparation</strong>
        The dataset consists of structured tables containing traffic collision data. Before conducting any analysis, it is essential to ensure that the data is properly formatted and structured for efficient processing.
        Check for data consistency and ensure all columns are correctly formatted.
        Apply sampling techniques if needed to extract a representative subset for analysis.
        Structure and prepare the data for further processing and analysis.
    
    <li>
        <strong>Data Cleaning</strong>
            2.1 <strong>Fixing Columns:</strong> Ensure all columns are properly named and formatted.
            2.2 <strong>Handling Missing Values:</strong> Decide on an approach to handle missing data (e.g., imputation or removal). Mention the approach in your report.
            2.3 <strong>Handling Outliers:</strong> Identify outliers in the dataset and explain why they are considered outliers. It is not necessary to remove them for this task, but mention your approach for handling them.
    
    <li>
        <strong>Exploratory Data Analysis</strong>
        Finding Patterns and analyze the dataset and find patterns based on the following points:
                <ul>
                3.1 Classify variables into categorical and numerical types.
                3.2 Analyze the distribution of collision severity.
                3.3 Examine weather conditions during collisions.
                3.4 Analyze the distribution of victim ages.
                3.5 Study the relationship between collision severity and the number of victims.
                3.6 Analyze the correlation between weather conditions and collision severity.
                3.7 Visualize the impact of lighting conditions on collision severity.
                3.8 Extract and analyze weekday-wise collision trends.
                3.9 Assess the number of collisions occurring on different days of the week.
                3.10 Study spatial distribution of collisions by county.
                3.11 Generate a scatter plot to analyze collision locations geographically.
                3.12 Extract and analyze collision trends over time, including yearly, monthly, and hourly trends.
</ul>

<li>
<strong>ETL Querying</strong>
Write PySpark SQL queries for the following:
<ol>
    4.1. Identify the top 5 counties with the highest number of collisions.
    4.2. Identify the month with the highest number of collisions.
    4.3. Determine the most common weather condition during collisions.
    4.4. Calculate the percentage of collisions that resulted in fatalities.
    4.5. Find the most dangerous time of day for collisions.
    4.6. Identify the top 5 road surface conditions with the highest collision frequency.
    4.7. Analyze lighting conditions that contribute to the highest number of collisions.
</ol>

<li>
<strong>Conclusion</strong>
Provide final insights and recommendations based on the analysis:
    <ul>
        5.1 Recommendations to improve road safety by identifying high-risk locations and peak accident times for infrastructure improvements.
        5.2 Suggestions to optimize traffic management by analyzing trends in collision severity, weather conditions, and lighting to improve road design and traffic signal timing.
        5.3 Propose data-driven policy changes to enhance pedestrian and cyclist safety based on collision trends involving vulnerable road users.
        5.4 Identify potential high-risk zones for proactive intervention by examining geographic collision density and historical accident data.
        5.5 Assess the impact of environmental factors such as weather, road surface conditions, and lighting on accident frequency and severity.
        5.6 Develop predictive models to anticipate collision hotspots and support proactive safety measures.
        </ul>
        Conclude the analysis by summarizing key findings and business implications.
        Explain the results of univariate, segmented univariate, and bivariate analyses in real-world traffic safety and policy terms.
        Include visualizations and summarize the most important results in the report. Insights should explain why each variable is important and how they can influence traffic safety policies and urban planning.
        </ul>
        
    <li>
    <strong>Visualization Integration [Optional]</strong>
    <p>Enhance the project by incorporating a visualization component that connects the processed data stored in an S3 bucket to a business intelligence tool such as Tableau or Power BI. This involves setting up the connection between the S3 bucket and the chosen visualization tool, importing the processed dataset for analysis and visualization, creating interactive dashboards to explore key trends and insights and ensuring data updates are reflected dynamically in the visualization tool.

</ol>

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')

Install Required Libraries

In [ ]:
## Install the required libraries
# !pip install --quiet pyspark==3.5.4 pandas==2.2.2

In [ ]:
# Import the necessary libraries
import sqlite3
import pandas as pd
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col,
    sum,
    to_date,
    when,
    year,
    month,
    to_timestamp,
    hour,
    date_format
)
from pyspark.sql.types import DateType
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns


from pyspark.sql.types import StringType
from pyspark.ml.feature import StringIndexer

import calendar



# 1. Data Preparation
The dataset consists of structured tables containing traffic collision data.

Before conducting any analysis, it is essential to ensure that the data is properly formatted and structured for efficient processing.

Check for data consistency and ensure all columns are correctly formatted.

In [ ]:
# Write code to load the data and check the schema
spark = SparkSession.builder.appName("collision-assesment").getOrCreate()

dfs = {}
for file in Path("./Crash_Data_Analysis_Dataset/").glob("sample_*.csv"):
    file_key = file.stem.removeprefix("sample_")
    dfs[file_key] = (
        spark.read
        .option("header", True)
        .option("inferSchema", True)
        .csv(str(file))
    )
dfs.keys()

# 2. Data Cleaning

In [ ]:
#Display Schema & Sample Data


In [ ]:
for name, df in dfs.items():
    print(f"Column names of {name}:")
    print(df.columns)
    print('==================')

In [ ]:
dfs["collisions"].show(1, vertical=True)

In [ ]:
print(dfs["victims"].show(1, vertical=True))

In [ ]:
print(dfs["parties"].show(1, vertical=True))

In [ ]:
print(dfs["case_ids"].show(1, vertical=True))

## 2.1 Missing Values

In [ ]:
#Check for Missing Values
def get_missing_values(df):
    total_rows = df.count()

    missing_df = (
        df.select([
            sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
            for c in df.columns
        ])
        .toPandas()
        .T
        .reset_index()
    )
    
    missing_df.columns = ["Column", "missing values"]
    
    missing_df['missing percent'] = (
        missing_df['missing values'] / total_rows * 100
    ).round(2)
    
    missing_df = (
        missing_df.sort_values(
            by="missing percent",
            ascending=False
        )
        .reset_index(drop=True)
    )


    missing_df = (
        missing_df[missing_df["missing values"] > 0]
        .sort_values('missing percent', ascending = False)
        .reset_index(drop= True)
    )
    return missing_df
    

In [ ]:
# Work on collision data
collision_missing = get_missing_values(dfs["collisions"])
display(collision_missing)

In [ ]:
# Work on victims data
collision_missing = get_missing_values(dfs["victims"])
display(collision_missing)

In [ ]:
# Sort from highest to lowest missing percentage
plot_df = collision_missing.sort_values(
    by="missing percent",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_df["Column"],
    plot_df["missing percent"]
)

plt.xlabel("Missing Percentage (%)")
plt.ylabel("Columns")
plt.title("Missing Values in Victims Dataset")

plt.tight_layout()
plt.show()

In [ ]:
# Work on parties data
collision_missing = get_missing_values(dfs["parties"])
display(collision_missing)

In [ ]:
# Sort from highest to lowest missing percentage
plot_df = collision_missing.sort_values(
    by="missing percent",
    ascending=True
)

plt.figure(figsize=(10, 6))

plt.barh(
    plot_df["Column"],
    plot_df["missing percent"]
)

plt.xlabel("Missing Percentage (%)")
plt.ylabel("Columns")
plt.title("Missing Values in Parties Dataset")

plt.tight_layout()
plt.show()

In [ ]:
# Work on case_ids data
collision_missing = get_missing_values(dfs["case_ids"])
display(collision_missing)

In [ ]:
#Drop Sparse Columns
# No column was removed as each column hold some value

In [ ]:
dfs['collisions'].select('collision_time').show()

In [ ]:
#Convert Data Types
 # collision date columns 'collision_date', 'collision_time', 'process_date'
dfs['collisions'] = (
    dfs['collisions']
    .withColumn(
        'collision_date',
        to_date(col('collision_date'), 'yyyy-MM-dd')
    )
    .withColumn(
        'collision_time',
        to_date(col('collision_time'), 'yyyy-MM-dd')
    )
    .withColumn(
        'process_date',
        to_date(col('process_date'), 'yyyy-MM-dd')
    )
)
dfs['collisions'].select(
    'collision_date',
    'collision_time',
    'process_date'
).printSchema()

In [ ]:
#Convert Data Types
 # collision parties 'vehicle_year'
dfs['parties'] = (
    dfs['parties']
    .withColumn(
        'vehicle_year',
        to_date(col('vehicle_year'), 'yyyy-MM-dd')
    )
)
dfs['parties'].select(
    'vehicle_year'
).printSchema()

In [ ]:
#Convert Data Types
 # collision case_ids 'db_year'
dfs['case_ids'] = (
    dfs['case_ids']
    .withColumn(
        'db_year',
        to_date(col('db_year'), 'yyyy-MM-dd')
    )
)
dfs['case_ids'].select(
    'db_year'
).printSchema()

In [ ]:
#Handle Missing Values

categorical_fill = {
    "weather_1": "Unknown",
    "road_surface": "Unknown",
    "road_condition_1": "Unknown",
    "lighting": "Unknown",
    "control_device": "Unknown",
    "type_of_collision": "Unknown",
    "motor_vehicle_involved_with": "Unknown",
    "primary_collision_factor": "Unknown",
    "pcf_violation_category": "Unknown",
    "pcf_violation": "Unknown",
    "special_condition": "Unknown"
}

dfs["collisions"] = dfs["collisions"].fillna(categorical_fill)

In [ ]:
numeric_fill = {
    "killed_victims": 0,
    "injured_victims": 0
}

dfs["collisions"] = dfs["collisions"].fillna(numeric_fill)

In [ ]:
victim_fill = {
    "victim_sex": "Unknown"
}

dfs["victims"] = dfs["victims"].fillna(victim_fill)

## 2.2 Fixing Columns

In [ ]:
#Remove Duplicates
for name, df in dfs.items():
    total_rows = df.count()
    unique_rows = df.dropDuplicates().count()

    print(f"{name}")
    print(f"Total Rows      : {total_rows}")
    print(f"Duplicate Rows  : {total_rows - unique_rows}")
    print("-" * 50)

In [ ]:
dfs["case_ids"] = dfs["case_ids"].dropDuplicates()

In [ ]:
#Detect Outliers using IQR

# List of numerical columns to check for outliers

collision_cols = [
    "distance",
    "party_count",
    "killed_victims",
    "injured_victims",
    "severe_injury_count",
    "other_visible_injury_count",
    "complaint_of_pain_injury_count"
]

victim_cols = [
    "victim_age"
]

party_cols = [
    "party_age",
    "vehicle_year",
    "party_number_killed",
    "party_number_injured"
]



In [ ]:
dfs["victims"].select("victim_age").printSchema()

## 2.3 Outlier Analysis

In [ ]:
def detect_outliers_iqr(df, column):

    # Remove null values
    clean_df = df.filter(col(column).isNotNull())

    # Handle DateType columns
    if isinstance(clean_df.schema[column].dataType, DateType):
        clean_df = clean_df.withColumn(
            "__value__",
            year(col(column))
        )
        target_col = "__value__"
    else:
        target_col = column

    # Calculate quartiles
    quantiles = clean_df.approxQuantile(
        target_col,
        [0.25, 0.75],
        0.01
    )

    if len(quantiles) < 2:
        print(f"Skipping '{column}' (unable to compute quartiles)")
        return None

    q1, q3 = quantiles

    # Calculate IQR
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    # Find outliers
    outlier_df = clean_df.filter(
        (col(target_col) < lower_bound) |
        (col(target_col) > upper_bound)
    )

    return {
        "Column": column,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "Lower Bound": lower_bound,
        "Upper Bound": upper_bound,
        "Outlier Count": outlier_df.count()
    }

In [ ]:
for column in collision_cols:
    result = detect_outliers_iqr(dfs["collisions"], column)

    if result:
        print(result)

In [ ]:
dfs["collisions"] \
    .select("distance") \
    .distinct() \
    .orderBy(col("distance").desc()) \
    .show(20, truncate=False)

In [ ]:
dfs["collisions"] = dfs["collisions"].filter(
    col("distance") <= 10000
)

In [ ]:
for column in collision_cols:
    print(f"\n{'=' * 60}")
    print(f"Column: {column}")

    dfs["collisions"] \
        .select(column) \
        .orderBy(col(column).desc()) \
        .show(20, truncate=False)

In [ ]:
for column in victim_cols:
    result = detect_outliers_iqr(dfs["victims"], column)

    if result:
        print(result)

In [ ]:
dfs["victims"] \
    .select("victim_age") \
    .orderBy(col("victim_age").desc()) \
    .show(30, truncate=False)

In [ ]:
dfs["victims"] = dfs["victims"].filter(
    col("victim_age") != 999
)

In [ ]:
dfs["victims"] = dfs["victims"].filter(
    (col("victim_age") >= 0) &
    (col("victim_age") <= 120)
)

In [ ]:
dfs["victims"] \
    .groupBy("victim_age") \
    .count() \
    .orderBy(col("victim_age").desc()) \
    .show(30, truncate=False)

In [ ]:
for column in party_cols:
    result = detect_outliers_iqr(dfs["parties"], column)

    if result:
        print(result)

In [ ]:
dfs["parties"] \
    .groupBy("party_age") \
    .count() \
    .orderBy(col("party_age").desc()) \
    .show(30, truncate=False)

In [ ]:
# party_number_killed
dfs["parties"] \
    .groupBy("party_number_killed") \
    .count() \
    .orderBy(col("party_number_killed").desc()) \
    .show(20)

# party_number_injured
dfs["parties"] \
    .groupBy("party_number_injured") \
    .count() \
    .orderBy(col("party_number_injured").desc()) \
    .show(20)

# 3. Exploratory Data Analysis

## 3.1.1. Data Preparation
Classify variables into categorical and numerical.

In [ ]:
# Encode Categorical Variables

# String Indexing for Categorical Columns
categorical_cols = [
    field.name
    for field in dfs["collisions"].schema.fields
    if isinstance(field.dataType, StringType)
]

print(categorical_cols)

In [ ]:
for column in categorical_cols:
    indexer = StringIndexer(
        inputCol=column,
        outputCol=f"{column}_index",
        handleInvalid="keep"
    )

    dfs["collisions"] = indexer.fit(dfs["collisions"]).transform(dfs["collisions"])

In [ ]:
dfs["collisions"].select(
    categorical_cols[:3] +
    [f"{c}_index" for c in categorical_cols[:3]]
).show(5)

In [ ]:
# Reordering & Renaming Columns
dfs["collisions"] = (
    dfs["collisions"]
    .withColumnRenamed("weather_1", "weather")
    .withColumnRenamed("road_condition_1", "road_condition")
)

In [ ]:
# Final cleaned data
# Final cleaned collisions dataset
cleaned_collisions = dfs["collisions"].select(
    "case_id",
    "collision_date",
    "collision_time",
    "collision_severity",
    "county_location",
    "county_city_location",
    "primary_road",
    "secondary_road",
    "weather",
    "lighting",
    "road_condition",
    "party_count",
    "killed_victims",
    "injured_victims",
    *[
        c for c in dfs["collisions"].columns
        if c not in [
            "case_id",
            "collision_date",
            "collision_time",
            "collision_severity",
            "county_location",
            "county_city_location",
            "primary_road",
            "secondary_road",
            "weather",
            "lighting",
            "road_condition",
            "party_count",
            "killed_victims",
            "injured_victims",
        ]
    ]
)

In [ ]:
cleaned_victims = dfs["victims"].select(
    "case_id",
    "id",
    "party_number",
    "victim_role",
    "victim_age",
    "victim_sex",
    "victim_degree_of_injury",
    "victim_seating_position",
    "victim_safety_equipment_1",
    "victim_safety_equipment_2",
    "victim_ejected"
)

In [ ]:
cleaned_parties = dfs["parties"].select(
    "case_id",
    "id",
    "party_number",
    "party_type",
    "at_fault",
    "party_age",
    "party_sex",
    "party_race",
    "party_sobriety",
    "direction_of_travel",
    "movement_preceding_collision",
    "vehicle_make",
    "vehicle_year",
    "statewide_vehicle_type",
    "chp_vehicle_type_towing",
    "chp_vehicle_type_towed",
    "party_number_killed",
    "party_number_injured",
    "financial_responsibility",
    "cellphone_in_use",
    "cellphone_use_type",
    "party_safety_equipment_1",
    "party_safety_equipment_2",
    "other_associate_factor_1"
)

In [ ]:
cleaned_case_ids = dfs["case_ids"].select(
    "case_id",
    "db_year"
)

In [ ]:
dfs["victims"] = cleaned_victims
dfs["parties"] = cleaned_parties
dfs["case_ids"] = cleaned_case_ids
dfs["collisions"] = cleaned_collisions

Loading the Final Cleaned Dataset into S3 Bucket

## 3.1.2. Analyze the distribution of collision severity.
Analyze the distribution of collision severity.

In [ ]:
# Univariate Analysis

# Collision Severity Distribution
severity_df = (
    cleaned_collisions
    .groupBy("collision_severity")
    .count()
    .orderBy("count", ascending=False)
)

# Convert to Pandas
severity_pd = severity_df.toPandas()
severity_pd
# Plot

plt.figure(figsize=(10, 6))

sns.barplot(
    data=severity_pd,
    x="collision_severity",
    y="count",
    palette="viridis"
)

plt.title("Distribution of Collision Severity", fontsize=16)
plt.xlabel("Collision Severity")
plt.ylabel("Number of Collisions")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

## 3.1.3. Weather conditions during collisions.
Examine weather conditions during collisions.

In [ ]:
# Weather Conditions During Collisions
weather_df = (
    cleaned_collisions
    .groupBy("weather")
    .count()
    .orderBy("count", ascending=False)
)
# Convert to Pandas
weather_pd = weather_df.toPandas()

weather_pd
# Plot

plt.figure(figsize=(10, 6))

sns.barplot(
    data=weather_pd,
    y="weather",
    x="count",
    palette="viridis"
)

plt.title("Weather Conditions During Collisions")
plt.xlabel("Number of Collisions")
plt.ylabel("Weather Condition")

plt.tight_layout()
plt.show()

## 3.1.4. Victime Age Distribution.
Analyze the distribution of victim ages.

In [ ]:
# Distribution of Victim Ages
victim_age_pd = (
    cleaned_victims
    .select("victim_age")
    .dropna()
    .toPandas()
)
# Convert to Pandas

# Plot
plt.figure(figsize=(10, 6))

sns.histplot(
    data=victim_age_pd,
    x="victim_age",
    bins=20,
    kde=True,
    color="steelblue"
)

plt.title("Distribution of Victim Ages")
plt.xlabel("Victim Age")
plt.ylabel("Number of Victims")

plt.tight_layout()
plt.show()

## 3.1.5. Collision Severity vs Number of Victims.
Study the relationship between collision severity and the number of victims.

In [ ]:
# Bivariate Analysis
# Total victims involved in each collision
severity_vs_victims = (
    cleaned_collisions
    .groupBy("collision_severity")
    .sum("injured_victims", "killed_victims")
    .withColumnRenamed("sum(injured_victims)", "injured_victims")
    .withColumnRenamed("sum(killed_victims)", "killed_victims")
    .toPandas()
)

severity_vs_victims["total_victims"] = (
    severity_vs_victims["injured_victims"] +
    severity_vs_victims["killed_victims"]
)

severity_vs_victims = severity_vs_victims.sort_values("collision_severity")
# Collision Severity vs. Number of Victims

# Convert to Pandas

# Plot
plt.figure(figsize=(10, 6))

sns.barplot(
    data=severity_vs_victims,
    x="collision_severity",
    y="total_victims",
    palette="viridis"
)

plt.title("Collision Severity vs Total Victims")
plt.xlabel("Collision Severity")
plt.ylabel("Total Victims")
plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

## 3.1.6. Weather Conditions vs Collision Severity.
Analyze the correlation between weather conditions and collision severity.

In [ ]:
weather_vs_severity = (
    cleaned_collisions
    .groupBy("weather", "collision_severity")
    .count()
    .toPandas()
)

pivot_df = weather_vs_severity.pivot(
    index="weather",
    columns="collision_severity",
    values="count"
).fillna(0)

# Convert each row to percentages
pivot_pct = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100

plt.figure(figsize=(12, 6))

pivot_pct.plot(
    kind="bar",
    stacked=True,
    figsize=(12, 6),
    colormap="viridis"
)

plt.title("Collision Severity Distribution by Weather")
plt.xlabel("Weather Condition")
plt.ylabel("Percentage of Collisions (%)")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Collision Severity")
plt.tight_layout()

plt.show()

## 3.1.7. Lighting conditions vs Collision Severity.
Visualize the impact of lighting conditions on collision severity.

In [ ]:

# Convert to Pandas
lighting_severity = (
    cleaned_collisions
    .groupBy("lighting", "collision_severity")
    .count()
    .toPandas()
)

# Pivot data for plotting
lighting_pivot = lighting_severity.pivot(
    index="lighting",
    columns="collision_severity",
    values="count"
).fillna(0)

# Order by total collisions
lighting_pivot = lighting_pivot.loc[
    lighting_pivot.sum(axis=1).sort_values(ascending=False).index
]

# Plot
lighting_pivot.plot(
    kind="bar",
    stacked=True,
    figsize=(10, 6)
)

plt.title("Lighting Conditions vs Collision Severity")
plt.xlabel("Lighting Condition")
plt.ylabel("Number of Collisions")
plt.xticks(rotation=45, ha="right")
plt.legend(title="Collision Severity")
plt.tight_layout()
plt.show()

## 3.1.8. Weekday-Wise Collision Trends.
Extract and analyze weekday-wise collision trends.

In [ ]:
# Extract the weekday


# Extract weekday
weekday_df = (
    cleaned_collisions
    .withColumn("weekday", date_format("collision_date", "EEEE"))
)

# Count collisions by weekday
weekday_counts = (
    weekday_df
    .groupBy("weekday")
    .count()
    .toPandas()
)
# Plot

weekday_order = [
    "Monday",
    "Tuesday",
    "Wednesday",
    "Thursday",
    "Friday",
    "Saturday",
    "Sunday"
]

weekday_counts["weekday"] = pd.Categorical(
    weekday_counts["weekday"],
    categories=weekday_order,
    ordered=True
)

weekday_counts = weekday_counts.sort_values("weekday")

plt.figure(figsize=(9, 5))
sns.barplot(
    data=weekday_counts,
    x="weekday",
    y="count"
)

plt.title("Collisions by Day of Week")
plt.xlabel("Day of Week")
plt.ylabel("Number of Collisions")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## 3.1.9. Spatial Distribution of Collisions.
Study spatial distribution of collisions by county.

In [ ]:
# Collision Density by County

# Convert to Pandas
county_df = (
    cleaned_collisions
    .groupBy("county_location")
    .count()
    .orderBy("count", ascending=False)
    .toPandas()
)

# Plot
plt.figure(figsize=(10, 12))

sns.barplot(
    data=county_df,
    y="county_location",
    x="count",
    palette="viridis"
)

plt.title("Collision Density by County")
plt.xlabel("Number of Collisions")
plt.ylabel("County")
plt.tight_layout()
plt.show()

## 3.1.10. Collision Analysis by Geography.
Generate a scatter plot to analyze collision locations geographically.

In [ ]:
# Scatter Plot of Collision Locations

# Convert PySpark DataFrame to Pandas, handling potential missing values
location_df = (
    cleaned_collisions
    .select("latitude", "longitude")
    .dropna()
    .toPandas()
)
# Convert to numeric (if needed) and handle invalid data


location_df["latitude"] = pd.to_numeric(location_df["latitude"], errors="coerce")
location_df["longitude"] = pd.to_numeric(location_df["longitude"], errors="coerce")

location_df = location_df.dropna()

# Keep only valid California coordinates
location_df = location_df[
    (location_df["latitude"].between(32, 43)) &
    (location_df["longitude"].between(-125, -114))
]

# Plot the scatter plot

plt.figure(figsize=(10, 10))

plt.scatter(
    location_df["longitude"],
    location_df["latitude"],
    s=1,
    alpha=0.2
)

plt.title("Collision Locations in California")
plt.xlabel("Longitude")
plt.ylabel("Latitude")

plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()


## 3.1.11. Collision Trends Over Time.
Extract and analyzing collision trends over time.

In [ ]:
# Extract year and month from collision_date

# Extract year and month from collision_date
monthly_trend = (
    cleaned_collisions
    .withColumn("year", year(col("collision_date")))
    .withColumn("month", month(col("collision_date")))
    .groupBy("year", "month")
    .count()
    .orderBy("year", "month")
    .toPandas()
)
# Plot

monthly_trend["date"] = pd.to_datetime(
    monthly_trend[["year", "month"]].assign(day=1)
)

plt.figure(figsize=(14, 6))

sns.lineplot(
    data=monthly_trend,
    x="date",
    y="count",
    marker="o"
)

plt.title("Monthly Collision Trend")
plt.xlabel("Year")
plt.ylabel("Number of Collisions")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

Analyze yearly, monthly and hourly trends in collisions.

In [ ]:
# Yearly Trend of Collisions

# Extract year and count collisions
yearly_trend = (
    cleaned_collisions
    .withColumn("year", year(col("collision_date")))
    .groupBy("year")
    .count()
    .orderBy("year")
    .toPandas()
)
# Plot

plt.figure(figsize=(10, 6))

sns.barplot(
    data=yearly_trend,
    x="year",
    y="count",
    palette="viridis"
)

plt.title("Yearly Trend of Collisions")
plt.xlabel("Year")
plt.ylabel("Number of Collisions")

plt.xticks(rotation=45)
plt.tight_layout()

plt.show()

In [ ]:
# Monthly Trend of Collisions

# Extract month and count collisions
monthly_trend = (
    cleaned_collisions
    .withColumn("month", month(col("collision_date")))
    .groupBy("month")
    .count()
    .orderBy("month")
    .toPandas()
)


monthly_trend["month_name"] = monthly_trend["month"].apply(
    lambda x: calendar.month_name[x]
)
# Plot
plt.figure(figsize=(10, 6))

sns.lineplot(
    data=monthly_trend,
    x="month_name",
    y="count",
    marker="o"
)

plt.title("Monthly Trend of Collisions")
plt.xlabel("Month")
plt.ylabel("Number of Collisions")
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Hourly Trend of Collisions
hourly_trend = (
    cleaned_collisions
    .withColumn("hour", hour(col("collision_time")))
    .groupBy("hour")
    .count()
    .orderBy("hour")
    .toPandas()
)
# Plot

plt.figure(figsize=(12, 6))

sns.lineplot(
    data=hourly_trend,
    x="hour",
    y="count",
    marker="o"
)

plt.title("Hourly Trend of Collisions")
plt.xlabel("Hour of Day")
plt.ylabel("Number of Collisions")

plt.xticks(range(0, 24))
plt.grid(alpha=0.3)

plt.tight_layout()
plt.show()

# 4. ETL Querying

## **4.1. Top 5 Counties** 

Identify the top 5 counties with the highest number of collisions.

In [ ]:
# Query: Identify the top 5 counties with the most collisions
# Query: Identify the top 5 counties with the most collisions

top_5_counties = (
    cleaned_collisions
    .groupBy("county_location")
    .count()
    .orderBy(col("count").desc())
    .limit(5)
)

top_5_counties.show(truncate=False)

## **4.2. Month with Highest Collisions** 

Identify the month with the highest number of collisions.

In [ ]:
# Query: Find the month with the highest number of collisions
# Query: Find the month with the highest number of collisions

monthly_collisions = (
    cleaned_collisions
    .withColumn("month", month("collision_date"))
    .groupBy("month")
    .count()
    .orderBy(col("count").desc())
)

monthly_collisions.show()

## **4.3. Weather Conditions with Highest Collisions.** 

Determine the most common weather condition during collisions.

In [ ]:
# Query: Find the most common weather condition during collisions

weather_collisions = (
    cleaned_collisions
    .groupBy("weather")
    .count()
    .orderBy(col("count").desc())
)

weather_collisions.show(truncate=False)

## **4.4. Fatal Collisions.** 

Calculate the percentage of collisions that resulted in fatalities.

In [ ]:
# Query: Determine the percentage of collisions that resulted in fatalities
total_collisions = cleaned_collisions.count()

fatal_collisions = (
    cleaned_collisions
    .filter(col("killed_victims") > 0)
    .count()
)

fatal_percentage = (fatal_collisions / total_collisions) * 100

print(f"Total Collisions      : {total_collisions}")
print(f"Fatal Collisions      : {fatal_collisions}")
print(f"Fatal Collision Rate  : {fatal_percentage:.2f}%")

## **4.5. Dangerous Time for Collisions.** 

Find the most dangerous time of day for collisions.

In [ ]:
# Query: Find the most dangerous time of day for collisions

cleaned_collisions.select("collision_time") \
    .distinct() \
    .orderBy("collision_time") \
    .show(20, truncate=False)

## **4.6. Road Surface Conditions.** 

Identify the top 5 road surface conditions with the highest collision frequency.

In [ ]:
# Query: List the top 5 road types with the highest collision frequency

road_conditions = (
    cleaned_collisions
    .groupBy("road_condition")
    .count()
    .orderBy(col("count").desc())
    .limit(5)
)

road_conditions.show(truncate=False)

## **4.7. Lighting Conditions.** 

Analyze lighting conditions that contribute to the highest number of collisions.

In [ ]:
# Query: Find the top 3 lighting conditions that lead to the most collisions
lighting_conditions = (
    cleaned_collisions
    .groupBy("lighting")
    .count()
    .orderBy(col("count").desc())
)

lighting_conditions.show(truncate=False)

#5. Conclusion 

Summarize the key findings, limitations, and practical implications of the analysis.

In this project, I developed an ETL pipeline to clean and prepare the California traffic collision dataset for analysis. The cleaning process included handling missing values, removing invalid records, correcting inconsistent values, renaming and reordering columns, and converting data into a format suitable for querying and visualization.

The analysis revealed several useful insights. 
- Los Angeles County recorded the highest number of collisions, followed by Orange, San Bernardino, San Diego, and Riverside counties.
- October had the highest number of reported collisions, while July recorded the lowest among all months.
- Clear weather accounted for the majority of collisions, followed by cloudy and rainy conditions.
- Normal road surface conditions and daylight were associated with the highest number of reported collisions, mainly because these are the conditions under which most vehicles are on the road.
- The victim age distribution became much more realistic after removing invalid age values, with most victims belonging to the young and middle-aged population.
- Most collisions resulted in property damage only, while severe injury collisions represented only a small proportion of the total cases.

During the ETL process, several data quality issues were identified. Invalid age values such as 999 years and unrealistic ages above 120 years were removed from the victims and parties datasets. Missing categorical values were replaced with **"Unknown"**, and inconsistent column names were standardized to improve readability. One important limitation of the dataset was the collision time field. Although the dataset contained a `collision_time` column, it did not contain valid hour-level information, making it impossible to determine the most dangerous time of day for collisions. Rather than producing misleading results, this limitation was documented in the report.

Another observation was that no fatal collisions were identified using the `killed_victims` field after the cleaning process. This indicates either that fatalities are not represented in the provided dataset or that the field does not contain complete information for this analysis. This limitation should be considered when interpreting the results.

Overall, the ETL pipeline produced a clean and consistent dataset that can be used for further analysis and visualization. More importantly, the project demonstrated that careful data cleaning is essential before drawing conclusions, as inaccurate or incomplete data can significantly affect the reliability of analytical results.

#6. Visualization Integration using Tableau/ PowerBI <font color = red>[Optional]</font> 